In [16]:
!pip install -U altair vega_datasets

In [17]:
import altair as alt
import pandas as pd

df = pd.read_csv("/content/movies_metadata.csv")

/tmp/ipykernel_2349/3617515781.py:4: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/movies_metadata.csv")


In [18]:
df.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [19]:
# Google AI helped with the data cleaning. Expanding rows for the sake of aggregation.

import ast

# Converts belongs_to_collection to a standard dict.
df['belongs_to_collection'] = df['belongs_to_collection'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.strip() != "" else {})
# Gets the name of the collection.
df['belongs_to_collection'] = df['belongs_to_collection'].apply(lambda d: d.get('name', 'No Collection') if isinstance(d, dict) else 'No Collection')

# Converts genres to a standard list.
df['genres'] = df['genres'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.strip() != "" else [])
# Gets only the name of the genres and puts into the list.
df['genres'] = df['genres'].apply(lambda standard_list: [d['name'] for d in standard_list] if isinstance(standard_list, list) else [])
# Explodes the list.
df = df.explode('genres')

df['production_companies'] = df['production_companies'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.strip() != "" else [])
df['production_companies'] = df['production_companies'].apply(lambda standard_list: [d['name'] for d in standard_list] if isinstance(standard_list, list) else [])
df = df.explode('production_companies')

df['production_countries'] = df['production_countries'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.strip() != "" else [])
df['production_countries'] = df['production_countries'].apply(lambda standard_list: [d['name'] for d in standard_list] if isinstance(standard_list, list) else [])
df = df.explode('production_countries')

df['spoken_languages'] = df['spoken_languages'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.strip() != "" else [])
df['spoken_languages'] = df['spoken_languages'].apply(lambda standard_list: [d['name'] for d in standard_list] if isinstance(standard_list, list) else [])
df = df.explode('spoken_languages')

df.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,Toy Story Collection,30000000,Animation,http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,English,Released,NaN,Toy Story,False,7.7,5415.0
0,False,Toy Story Collection,30000000,Comedy,http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,English,Released,NaN,Toy Story,False,7.7,5415.0
0,False,Toy Story Collection,30000000,Family,http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,English,Released,NaN,Toy Story,False,7.7,5415.0
1,False,No Collection,65000000,Adventure,NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,English,Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
1,False,No Collection,65000000,Adventure,NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,Français,Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0


In [20]:
# Filter out 0-revenue rows.
df = df[df['revenue'] > 0]

# Group by median revenue for genre.

df_median = (
    df.groupby('genres')
    .agg(
        median_revenue=('revenue', 'median'),
        movie_count=('revenue', 'count')
    )
    .reset_index()
)


chart = (
    alt.Chart(df_median)
    .mark_bar(opacity=0.7)
    .encode(
        x=alt.X('median_revenue:Q', title='Median Revenue (USD)', axis=alt.Axis(format='$,.0f')), #Q for quantitative
        y=alt.Y("genres:N", title="Genre", sort='-x'), #N for nominal (categorical), Sorts by median revenue.
        color=alt.Color("genres:N", legend=None), # Removes the redundant color legend.
        tooltip=[alt.Tooltip("median_revenue:Q", title='Median Revenue (USD)', format='$,.0f'),
                 alt.Tooltip("genres:N", title='Genre'),
                 alt.Tooltip('count():Q', title='Total Movie Count')],
    )
    .properties(width=600, height=360, title="Median Movie Revenue by Genre")
    .interactive()

)
chart

alt.Chart(...)

In [21]:
# Group by average revenue for genre.

df_average = (
    df.groupby('genres')
    .agg(
        average_revenue=('revenue', 'mean'),
        movie_count=('revenue', 'count')
    )
    .reset_index()
)


chart2 = (
    alt.Chart(df_average)
    .mark_bar(opacity=0.7)
    .encode(
        x=alt.X('average_revenue:Q', title='Average Revenue (USD)', axis=alt.Axis(format='$,.0f')), #Q for quantitative
        y=alt.Y("genres:N", title="Genre", sort='-x'), #N for nominal (categorical), Sorts by average revenue.
        color=alt.Color("genres:N", legend=None), # Removes the redundant color legend.
        tooltip=[alt.Tooltip("average_revenue:Q", title='Average Revenue (USD)', format='$,.0f'),
                 alt.Tooltip("genres:N", title='Genre'),
                 alt.Tooltip('count():Q', title='Total Movie Count')],
    )
    .properties(width=600, height=360, title="Average Movie Revenue by Genre")
    .interactive()

)
chart2

alt.Chart(...)

In [25]:
# Group by median profit for genre.

# This is to make sure that the revenue and budget columns are numeric.
df['revenue'] = pd.to_numeric(df['revenue'], errors='coerce')
df['budget'] = pd.to_numeric(df['budget'], errors='coerce')

df = df[df['budget'] > 0]

df_median = (
    df.groupby('genres')
    .agg(
        median_revenue=('revenue', 'median'),
        median_budget=('budget', 'median'),
        movie_count=('revenue', 'count')
    )
    .reset_index()
)

df_median['median_profit'] = df_median['median_revenue'] - df_median['median_budget']

chart3 = (
    alt.Chart(df_median)
    .mark_bar(opacity=0.7)
    .encode(
        x=alt.X('median_profit:Q', title='Median Profit (USD)', axis=alt.Axis(format='$,.0f')), #Q for quantitative
        y=alt.Y("genres:N", title="Genre", sort='-x'), #N for nominal (categorical), Sorts by median profit.
        color=alt.Color("genres:N", legend=None), # Removes the redundant color legend.
        tooltip=[alt.Tooltip("median_profit:Q", title='Median Profit (USD)', format='$,.0f'),
                 alt.Tooltip("genres:N", title='Genre'),
                 alt.Tooltip('count():Q', title='Total Movie Count')],
    )
    .properties(width=600, height=360, title="Median Movie Profit by Genre")
    .interactive()

)
chart3

alt.Chart(...)

In [26]:
# Group by average revenue for genre.

df_average = (
    df.groupby('genres')
    .agg(
        average_revenue=('revenue', 'mean'),
        average_budget=('budget','mean'),
        movie_count=('revenue', 'count')
    )
    .reset_index()
)

df_average['average_profit'] = df_average['average_revenue'] - df_average['average_budget']


chart4 = (
    alt.Chart(df_average)
    .mark_bar(opacity=0.7)
    .encode(
        x=alt.X('average_profit:Q', title='Average Profit (USD)', axis=alt.Axis(format='$,.0f')), #Q for quantitative
        y=alt.Y("genres:N", title="Genre", sort='-x'), #N for nominal (categorical), Sorts by average profit.
        color=alt.Color("genres:N", legend=None), # Removes the redundant color legend.
        tooltip=[alt.Tooltip("average_profit:Q", title='Average Profit (USD)', format='$,.0f'),
                 alt.Tooltip("genres:N", title='Genre'),
                 alt.Tooltip('count():Q', title='Total Movie Count')],
    )
    .properties(width=600, height=360, title="Average Movie Profit by Genre")
    .interactive()

)
chart4

alt.Chart(...)